# Colab 10 — normLev Similarity, Variable-Length, Synthetic Prototype

**Goal of this notebook.** Validate the new architecture on synthetic variable-length AA sequences before moving to real CATH data. We are *not* running the cross-representation test here — that comes later, once the in-domain V1 plot looks right and the supervisor's CATH dataset arrives.

## What changed from colab9a

| | colab9a | colab10 |
|---|---|---|
| Length | fixed `SEQ_LEN=8` | variable, `MIN_LEN=40` to `MAX_LEN_SEQ=80`, padded to `MAX_LEN=96` |
| Vocab | 20 (AA only) | 21 (AA + `<PAD>`) |
| Label | `lev / SEQ_LEN` (distance, 0 = identical) | `1 - lev / max(\|a\|,\|b\|)` (similarity, 1 = identical) |
| Pooling | `flatten(1)` | `flatten(1)` with PAD positions zero-masked first |
| Loss | weighted MSE, `weight = exp(-α·d)` | plain MSE on similarity (iter 1) |
| Eval | planted-recall@k retrieval | V1 pointwise predicted-vs-true scatter |

## Why synthetic dummy data first

We want to validate three things independently of the real-data pipeline:
1. The pad-aware encoder doesn't degenerate — PAD masking actually works.
2. The model learns the new normLev label shape (similarity in [0, 1], 1 = identical).
3. The V1 plot shows a roughly diagonal cloud, ideally tight at the high-similarity end (top-right).

If V1 looks reasonable on synthetic data, we have confidence the architecture is correctly wired. We then swap in real CATH proteins (same code, just different data source) and add the V2 superfamily-separation eval.

## 1. Setup & Imports

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
!pip install torch torchvision --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Alphabet (with PAD), Levenshtein, normLev

PAD index is the highest in the vocab so it doesn't disturb the AA indices. Embedding gets `padding_idx=PAD_IDX`, which initializes the PAD row to zero and keeps it zero across training (PyTorch behavior of `padding_idx`).

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
VOCAB_SIZE = len(AA_ALPHABET) + 1   # +1 for <PAD>
PAD_IDX = len(AA_ALPHABET)          # PAD is index 20

CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}

# Sequence length parameters
MIN_LEN_SEQ = 40       # minimum length of a generated synthetic sequence
MAX_LEN_SEQ = 80       # maximum length of a generated synthetic sequence
MAX_LEN = 96           # padding length (headroom for ins-heavy perturbations)

rng = np.random.default_rng(SEED)

def random_seq(min_len=MIN_LEN_SEQ, max_len=MAX_LEN_SEQ):
    length = int(rng.integers(min_len, max_len + 1))
    return ''.join(rng.choice(list(AA_ALPHABET), size=length))

def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    idx = [CHAR_TO_IDX[c] for c in seq][:max_len]
    idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)

def levenshtein(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[m][n]

def norm_lev(a, b):
    """Similarity in [0, 1]. 1 = identical, 0 = max different."""
    L = max(len(a), len(b))
    if L == 0:
        return 1.0
    return 1.0 - levenshtein(a, b) / L

print(f'Vocab size (incl. PAD): {VOCAB_SIZE}, PAD index: {PAD_IDX}')
print(f'Sequence lengths: synthetic in [{MIN_LEN_SEQ}, {MAX_LEN_SEQ}], padded to {MAX_LEN}')

# Sanity check on normLev
a = 'ACDEFGHI'
b = 'ACDEFGHI'
print(f"normLev('{a}', '{b}') = {norm_lev(a, b):.3f}  (expect 1.0)")
b2 = 'ACDEFGHIKL'
print(f"normLev('{a}', '{b2}') = {norm_lev(a, b2):.3f}  (lev={levenshtein(a, b2)}, max_len={max(len(a), len(b2))})")
b3 = 'WYWYWYWY'
print(f"normLev('{a}', '{b3}') = {norm_lev(a, b3):.3f}  (low similarity)")

## 3. Synthetic corpus — variable-length AA sequences

Generate a corpus of variable-length sequences using a uniform AA distribution. This is *dummy data* — it has no biological structure, no superfamilies, no homologs. The point is purely to test that the architecture handles variable length + padding correctly and learns the normLev function.

In [ ]:
CORPUS_SIZE = 5000
corpus = list({random_seq() for _ in range(CORPUS_SIZE * 2)})[:CORPUS_SIZE]  # dedupe, then trim
print(f'Corpus size (deduped): {len(corpus)}')

lengths = [len(s) for s in corpus]
plt.figure(figsize=(8, 3))
plt.hist(lengths, bins=range(MIN_LEN_SEQ, MAX_LEN_SEQ + 2), edgecolor='k', alpha=0.75)
plt.xlabel('Sequence length')
plt.ylabel('Count')
plt.title('Synthetic corpus length distribution')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Pair generation

**Strategy.** We generate pairs from two sources:
1. **Perturbation pairs** (majority): pick a seed from the corpus, apply `k` random edits (sub / ins / del). True normLev is *measured* from the actual sequences afterward, not assumed equal to `k`.
2. **Random pairs** (minority): sample two independent corpus sequences. These are almost always low-similarity (uniform random AA pairs share little).

Why this mix: perturbation pairs give us mid-to-high similarity coverage; random pairs anchor the low-similarity end. Together they should yield a normLev distribution covering [0, 1] reasonably well.

**Variable-length aware perturbation.** No post-hoc re-pad/truncate. Length drifts with the operations. We just guard against `len == 0` (mass deletion) and against growing past `MAX_LEN`.

In [ ]:
def perturb(seq, k, max_len=MAX_LEN):
    s = list(seq)
    for _ in range(k):
        if len(s) == 0:
            op = 'ins'
        elif len(s) >= max_len:
            op = rng.choice(['sub', 'del'])  # don't grow further
        else:
            op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s))
            choices = [c for c in AA_ALPHABET if c != s[i]]
            s[i] = rng.choice(choices)
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1)
            s.insert(i, rng.choice(list(AA_ALPHABET)))
        elif op == 'del':
            i = rng.integers(0, len(s))
            del s[i]
    return ''.join(s)

N_PAIRS = 30000
PERTURB_FRAC = 0.8

pairs = []
attempts = 0
max_attempts = N_PAIRS * 4
while len(pairs) < N_PAIRS and attempts < max_attempts:
    attempts += 1
    if rng.random() < PERTURB_FRAC:
        seed = corpus[int(rng.integers(0, len(corpus)))]
        # k spans 0 to ~80% of seed length — covers the full similarity spectrum
        k = int(rng.integers(0, max(2, int(0.8 * len(seed)))))
        other = perturb(seed, k)
    else:
        i, j = rng.integers(0, len(corpus), size=2)
        seed, other = corpus[i], corpus[j]
    if len(seed) == 0 or len(other) == 0:
        continue
    if len(seed) > MAX_LEN or len(other) > MAX_LEN:
        continue
    label = norm_lev(seed, other)
    pairs.append((seed, other, label))

print(f'Generated {len(pairs)} training pairs (in {attempts} attempts).')
labels = np.array([p[2] for p in pairs])
print(f'normLev label stats: min={labels.min():.3f}, max={labels.max():.3f}, mean={labels.mean():.3f}')

plt.figure(figsize=(8, 3))
plt.hist(labels, bins=40, edgecolor='k', alpha=0.75)
plt.xlabel('normLev (1 = identical, 0 = max different)')
plt.ylabel('Count')
plt.title('Training-pair normLev distribution')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Dataset & DataLoader — padding to MAX_LEN

Each item is `(encoded_a, encoded_b, label)` where each encoded sequence is padded to `MAX_LEN` with `PAD_IDX`. The encoder will mask PAD positions before flatten.

In [ ]:
class PairDataset(Dataset):
    def __init__(self, pairs, max_len=MAX_LEN, pad_idx=PAD_IDX):
        self.pairs = pairs
        self.max_len = max_len
        self.pad_idx = pad_idx
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        a, b, label = self.pairs[i]
        return (
            encode_pad(a, self.max_len, self.pad_idx),
            encode_pad(b, self.max_len, self.pad_idx),
            torch.tensor(label, dtype=torch.float32),
        )

dataset = PairDataset(pairs)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

# Sanity: check a single batch shape
a_batch, b_batch, label_batch = next(iter(dataloader))
print(f'Batch shapes: a={tuple(a_batch.shape)}, b={tuple(b_batch.shape)}, labels={tuple(label_batch.shape)}')
print(f'PAD-positions in first sequence of batch: {(a_batch[0] == PAD_IDX).sum().item()} / {MAX_LEN}')

## 6. Pad-aware Siamese encoder

The architecture mirrors colab9a's flatten-based encoder, with two changes:
1. `nn.Embedding(VOCAB_SIZE, embed_dim, padding_idx=PAD_IDX)` — PAD's embedding is held at zero throughout training.
2. After the conv stack, we *zero out* PAD positions before flattening. The downstream Linear layer then sees zeros at PAD positions and learns to ignore them.

**Similarity output.** We map L2 distance between L2-normalized embeddings to a similarity in [0, 1] via `sim = 1 - L2_distance / 2` (max L2 distance on the unit hypersphere is 2). This makes the loss a direct MSE between predicted and true normLev.

In [ ]:
class SiameseEncoder(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32, hidden1=32, hidden2=64,
                 max_len=MAX_LEN, out_dim=128, pad_idx=PAD_IDX):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        self.fc = nn.Linear(hidden2 * max_len, out_dim)

    def forward(self, x):
        # x: (B, L), with PAD at trailing positions
        mask = (x != self.pad_idx).float()         # (B, L), 1 for real, 0 for pad
        x = self.embedding(x)                      # (B, L, embed_dim)
        x = x.permute(0, 2, 1)                     # (B, embed_dim, L)
        x = F.relu(self.conv1(x))                  # (B, hidden1, L)
        x = F.relu(self.conv2(x))                  # (B, hidden2, L)
        x = x * mask.unsqueeze(1)                  # zero PAD positions
        x = x.flatten(1)                           # (B, hidden2 * L)
        x = self.fc(x)                             # (B, out_dim)
        x = F.normalize(x, p=2, dim=1)             # unit hypersphere
        return x


class SiameseNetwork(nn.Module):
    def __init__(self, **kw):
        super().__init__()
        self.encoder = SiameseEncoder(**kw)
    def forward(self, a, b):
        ea = self.encoder(a)
        eb = self.encoder(b)
        l2 = torch.norm(ea - eb, p=2, dim=1)       # in [0, 2]
        sim = 1.0 - l2 / 2.0                        # in [0, 1]
        return sim
    def encode(self, x):
        return self.encoder(x)

model = SiameseNetwork().to(device)
print(model)
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

# Sanity: confirm a sequence equals itself in similarity
test = encode_pad(corpus[0]).unsqueeze(0).to(device)
with torch.no_grad():
    print(f'sim(seq, seq) before training = {model(test, test).item():.4f}  (expect 1.0)')

## 7. Training — plain MSE on normLev

Iteration 1 uses straight MSE. No U-shape weighting yet. We'll inspect the V1 plot and decide whether the high-similarity zone needs more emphasis.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 30
losses = []

model.train()
for epoch in range(1, num_epochs + 1):
    epoch_loss = 0.0
    nb = 0
    for a, b, label in dataloader:
        a = a.to(device); b = b.to(device); label = label.to(device)
        pred = model(a, b)
        loss = F.mse_loss(pred, label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        nb += 1
    avg = epoch_loss / nb
    losses.append(avg)
    if epoch % 2 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{num_epochs} — MSE: {avg:.5f}')

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.title('Training Loss — plain MSE on normLev')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f'Final MSE: {losses[-1]:.5f}')

## 8. V1 evaluation — predicted vs. true normLev

Build a held-out pair set from *fresh* synthetic sequences (not in the training corpus). Plot predicted similarity (y) against true normLev (x). The reference is `y = x`.

We report:
- **Pearson r** — linear fit quality across the whole range.
- **Spearman ρ** — rank correlation across the whole range.
- **Spearman ρ restricted to true normLev > 0.5** — the biologically relevant zone.

In [ ]:
# Held-out: fresh seeds + perturbations to ensure good label coverage.
N_HELDOUT = 5000
heldout = []
while len(heldout) < N_HELDOUT:
    if rng.random() < PERTURB_FRAC:
        seed = random_seq()
        k = int(rng.integers(0, max(2, int(0.8 * len(seed)))))
        other = perturb(seed, k)
    else:
        seed = random_seq()
        other = random_seq()
    if len(seed) == 0 or len(other) == 0:
        continue
    if len(seed) > MAX_LEN or len(other) > MAX_LEN:
        continue
    heldout.append((seed, other, norm_lev(seed, other)))

model.eval()
true_vals = []
pred_vals = []
with torch.no_grad():
    BATCH = 512
    for i in range(0, len(heldout), BATCH):
        batch = heldout[i:i+BATCH]
        a = torch.stack([encode_pad(p[0]) for p in batch]).to(device)
        b = torch.stack([encode_pad(p[1]) for p in batch]).to(device)
        pred = model(a, b).cpu().numpy()
        true_vals.extend([p[2] for p in batch])
        pred_vals.extend(pred.tolist())
true_vals = np.array(true_vals)
pred_vals = np.array(pred_vals)

pr, _ = pearsonr(true_vals, pred_vals)
sr, _ = spearmanr(true_vals, pred_vals)
mask_high = true_vals > 0.5
if mask_high.sum() > 10:
    sr_high, _ = spearmanr(true_vals[mask_high], pred_vals[mask_high])
else:
    sr_high = float('nan')

print(f'Pearson r (all):                        {pr:.4f}')
print(f'Spearman ρ (all):                       {sr:.4f}')
print(f'Spearman ρ (true normLev > 0.5):        {sr_high:.4f}  (high-similarity zone)')
print(f'Held-out pairs: {len(heldout)}, of which {int(mask_high.sum())} in high-similarity zone')

plt.figure(figsize=(7, 6))
plt.scatter(true_vals, pred_vals, alpha=0.1, s=5)
plt.plot([0, 1], [0, 1], 'r-', linewidth=2, label='y = x (perfect)')
plt.xlabel('True normLev')
plt.ylabel('Predicted normLev')
plt.title(f'V1: Predicted vs True normLev\nPearson r={pr:.3f}, Spearman ρ={sr:.3f}, ρ>0.5={sr_high:.3f}')
plt.xlim(-0.05, 1.05); plt.ylim(-0.05, 1.05)
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 9. Diagnostic — a few example pairs

Print a handful of held-out pairs with predicted vs true normLev. Useful sanity check for spot-checking failure modes.

In [ ]:
# Sort by true normLev and print 5 high, 5 mid, 5 low
idx_sorted = np.argsort(true_vals)
n = len(idx_sorted)
samples = list(idx_sorted[-5:]) + list(idx_sorted[n//2 - 2 : n//2 + 3]) + list(idx_sorted[:5])
print(f"{'true':>6}  {'pred':>6}  {'len_a':>6}  {'len_b':>6}  pair")
for i in samples:
    a, b, _ = heldout[i]
    print(f'{true_vals[i]:6.3f}  {pred_vals[i]:6.3f}  {len(a):>6}  {len(b):>6}  ({a[:25]}…, {b[:25]}…)')

## 10. Decision criteria — ready for real CATH data?

**Architecture-validation success criteria for this synthetic prototype:**
- Pearson r (all) > 0.85 — the model can fit the function on uniform random AA.
- Spearman ρ (true normLev > 0.5) > 0.7 — ordering is reasonable in the biologically relevant zone.
- V1 plot visually shows a roughly diagonal cloud, tight near `y = x` in the top-right corner.
- No catastrophic failure mode in the diagnostic sample (e.g., all predictions clustered at a single value).

**If passes →** swap data source to real CATH proteins (supervisor-provided), keep the rest of the pipeline. Add V2 same/different-superfamily separation eval. Then run AA↔SS cross-representation.

**If fails:**
- Inspect the diagnostic pairs — are length-mismatched pairs being handled? Are PAD-heavy pairs distorted?
- Try `embed_dim=64`, `out_dim=256` (richer per-token representation).
- Verify masking: print intermediate activations and confirm PAD positions are zero before flatten.
- Try smaller `MIN_LEN`/`MAX_LEN_SEQ` (e.g., 30–50) for faster iteration on the wiring.